In [6]:
!pip uninstall websockets

Found existing installation: websockets None


error: uninstall-no-record-file

× Cannot uninstall websockets None
╰─> The package's contents are unknown: no RECORD file was found for websockets.

hint: You might be able to recover from this via: pip install --force-reinstall --no-deps websockets==17.1


In [7]:
!pip install unsloth langchain langchain-core deepagents

  Using cached unsloth-2026.9.3-py3-none-any.whl.metadata (75 kB)
  Using cached langchain-1.4.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached deepagents-0.7.13-py3-none-any.whl.metadata (8.6 kB)
  Using cached unsloth_zoo-2026.9.2-py3-none-any.whl.metadata (33 kB)
  Using cached torch-2.12.1-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached torchvision-0.29.0-cp312-cp312-win_amd64.whl.metadata (5.7 kB)
  Using cached tyro-1.0.16-py3-none-any.whl.metadata (12 kB)
  Using cached xformers-0.0.35-py39-none-win_amd64.whl.metadata (1.0 kB)
  Using cached bitsandbytes-0.50.2-py3-none-win_amd64.whl.metadata (11 kB)
  Using cached triton_windows-3.8.0.post28-cp312-cp312-win_amd64.whl.metadata (1.1 kB)
  Using cached sentencepiece-0.2.2-cp312-cp312-win_amd64.whl.metadata (34 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.20.0-py3-none-any.whl.metadata (14 kB)
  Using cached 


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
error: uninstall-no-record-file

× Cannot uninstall websockets None
╰─> The package's contents are unknown: no RECORD file was found for websockets.

hint: You might be able to recover from this via: pip install --force-reinstall --no-deps websockets==17.1


In [8]:
from unsloth import FastLanguageModel

model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=8192,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

ModuleNotFoundError: No module named 'unsloth'

In [ ]:
import json
import re
from threading import Thread
from typing import Any, Dict, Iterator, List, Optional, Sequence, Union

import torch
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import (
    AIMessage,
    AIMessageChunk,
    BaseMessage,
    BaseMessageChunk,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_core.messages.tool import ToolCall
from langchain_core.outputs import ChatGeneration, ChatGenerationChunk, ChatResult
from langchain_core.tools import BaseTool
from langchain_core.utils.function_calling import convert_to_openai_tool
from transformers import TextIteratorStreamer


class UnslothChatModel(BaseChatModel):
    model: Any
    tokenizer: Any
    max_new_tokens: int = 1024
    temperature: float = 0.0
    bound_tools: List[Dict[str, Any]] = []

    @property
    def _llm_type(self) -> str:
        return "unsloth-chat-model"

    # ------------------------------------------------------------------
    # FIX 2: implementar bind_tools (antes ausente -> NotImplementedError)
    # ------------------------------------------------------------------
    def bind_tools(
        self,
        tools: Sequence[Union[Dict[str, Any], type, BaseTool]],
        *,
        tool_choice: Optional[Union[str, Dict[str, Any]]] = None,
        **kwargs: Any,
    ) -> "UnslothChatModel":
        """
        Modelos carregados via Unsloth/generate() não têm function-calling
        nativo. Aqui só guardamos a especificação das tools (em formato
        OpenAI) para poder injetá-las no prompt via `_convert_messages_to_prompt`
        e fazer um parsing simples da saída do modelo procurando chamadas
        de tool em JSON.
        """
        formatted_tools = [convert_to_openai_tool(t) for t in tools]
        return self.__class__(
            model=self.model,
            tokenizer=self.tokenizer,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            bound_tools=formatted_tools,
        )

    def _tools_system_snippet(self) -> str:
        if not self.bound_tools:
            return ""
        tools_desc = json.dumps(self.bound_tools, ensure_ascii=False, indent=2)
        return (
            "\n\nVocê tem acesso às seguintes ferramentas (tools):\n"
            f"{tools_desc}\n\n"
            "Se precisar usar uma ferramenta, responda APENAS com um JSON no "
            "formato: {\"tool_call\": {\"name\": \"<nome_da_tool>\", "
            "\"arguments\": {...}}}. Caso contrário, responda normalmente em "
            "texto."
        )

    def _convert_messages_to_prompt(self, messages: List[BaseMessage]) -> str:
        chat = []
        tools_snippet = self._tools_system_snippet()
        for m in messages:
            if isinstance(m, SystemMessage):
                content = m.content + tools_snippet
                tools_snippet = ""  # só injeta uma vez
                chat.append({"role": "system", "content": content})
            elif isinstance(m, HumanMessage):
                chat.append({"role": "user", "content": m.content})
            elif isinstance(m, AIMessage):
                chat.append({"role": "assistant", "content": m.content})
            elif isinstance(m, ToolMessage):
                chat.append(
                    {"role": "user", "content": f"[resultado da tool]: {m.content}"}
                )

        # se não havia SystemMessage nenhuma, garante que as tools apareçam
        if tools_snippet:
            chat.insert(0, {"role": "system", "content": tools_snippet.strip()})

        return self.tokenizer.apply_chat_template(
            chat, tokenize=False, add_generation_prompt=True
        )

    @staticmethod
    def _try_parse_tool_call(text: str) -> Optional[ToolCall]:
        match = re.search(r"\{.*\"tool_call\".*\}", text, re.DOTALL)
        if not match:
            return None
        try:
            data = json.loads(match.group(0))
            call = data.get("tool_call")
            if call and "name" in call:
                return ToolCall(
                    name=call["name"],
                    args=call.get("arguments", {}),
                    id="call_0",
                )
        except json.JSONDecodeError:
            return None
        return None

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> ChatResult:
        prompt = self._convert_messages_to_prompt(messages)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        # ------------------------------------------------------------
        # FIX 1: conflito entre max_new_tokens e max_length
        # ------------------------------------------------------------
        # O generation_config do modelo pode já trazer um `max_length`
        # herdado (ex.: igual ao max_seq_length usado no from_pretrained).
        # Isso conflita com `max_new_tokens` passado abaixo. Removemos o
        # `max_length` do generation_config (uma vez só) para deixar
        # apenas `max_new_tokens` como critério de parada.
        if getattr(self.model.generation_config, "max_length", None) is not None:
            self.model.generation_config.max_length = None

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                # não passar max_length aqui também, por segurança
                temperature=self.temperature if self.temperature > 0 else 0.01,
                do_sample=self.temperature > 0,
            )

        generated = output_ids[0][inputs["input_ids"].shape[-1]:]
        text = self.tokenizer.decode(generated, skip_special_tokens=True)

        tool_call = self._try_parse_tool_call(text) if self.bound_tools else None
        if tool_call:
            message = AIMessage(content="", tool_calls=[tool_call])
        else:
            message = AIMessage(content=text)

        generation = ChatGeneration(message=message)
        return ChatResult(generations=[generation])

    # ------------------------------------------------------------------
    # STREAM: emite tokens conforme são gerados (usado quando o código
    # chama `llm.stream(...)` ou `agent.stream(...)` em vez de `.invoke(...)`)
    # ------------------------------------------------------------------
    def _stream(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> Iterator[ChatGenerationChunk]:
        prompt = self._convert_messages_to_prompt(messages)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        # mesmo fix do problema 1: evita conflito max_new_tokens vs max_length
        if getattr(self.model.generation_config, "max_length", None) is not None:
            self.model.generation_config.max_length = None

        streamer = TextIteratorStreamer(
            self.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
        )

        generation_kwargs = dict(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature if self.temperature > 0 else 0.01,
            do_sample=self.temperature > 0,
            streamer=streamer,
        )

        # o .generate() é bloqueante, então roda numa thread separada
        # para poder consumir o streamer de forma incremental aqui.
        thread = Thread(target=self.model.generate, kwargs=generation_kwargs)
        thread.start()

        full_text = ""
        try:
            for token_text in streamer:
                if not token_text:
                    continue
                full_text += token_text
                chunk = ChatGenerationChunk(message=AIMessageChunk(content=token_text))
                if run_manager := kwargs.get("run_manager"):
                    run_manager.on_llm_new_token(token_text, chunk=chunk)
                yield chunk
        finally:
            thread.join()

        # se as tools estiverem "bindadas" e a resposta completa for uma
        # tool call, emite um chunk final com o tool_call estruturado
        # (o texto já foi streamado acima como conteúdo normal).
        if self.bound_tools:
            tool_call = self._try_parse_tool_call(full_text)
            if tool_call:
                yield ChatGenerationChunk(
                    message=AIMessageChunk(content="", tool_calls=[tool_call])
                )





In [ ]:
llm = UnslothChatModel(model=model, tokenizer=tokenizer, temperature=0)

In [ ]:
from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content="Olá, você está funcionando?")])
print(response.content)

Olá! Sim, estou funcionando. Como posso ajudar você hoje?


In [ ]:
from deepagents import create_deep_agent

In [ ]:
agent = create_deep_agent(
    tools=[],  # suas tools aqui
    model=llm,
    system_prompt="Você é um agent que planeja e executa tarefas passo a passo."
)

result = agent.invoke({"messages": [{"role": "user", "content": "Pesquise sobre X e resuma"}]})
print(result)

{'messages': [HumanMessage(content='Pesquise sobre X e resuma', additional_kwargs={}, response_metadata={}, id='3efb5114-d31a-4674-86b4-e566d1a127a3'), AIMessage(content='Posso pesquisar sobre qualquer coisa. Qual é o assunto sobre o qual você gostaria de saber mais?', additional_kwargs={}, response_metadata={}, id='lc_run--01a0653f-7d40-7b30-8004-0ac558e33a17-0', tool_calls=[], invalid_tool_calls=[])], 'files': {}}


In [ ]:
# streaming:
for chunk in llm.stream([HumanMessage(content="me fale sobre agentes de ia que usam os large language models")]):
    print(chunk.content, end="", flush=True)

Os agentes de IA que usam os Large Language Models (LLMs) são sistemas de inteligência artificial projetados para processar e gerar texto, similar ao que você está fazendo agora. Eles são baseados em redes neurais profundas treinadas em grandes conjuntos de dados de texto, o que lhes permite aprender padrões e relações entre as palavras e frases.

Os LLMs são projetados para realizar tarefas como:

1. **Geração de texto**: criar texto original, como respostas a perguntas, histórias, artigos ou até mesmo código de programação.
2. **Classificação de texto**: classificar texto em categorias, como classificar e-mails como spam ou não spam.
3. **Revisão de texto**: corrigir erros de ortografia, gramática e sintaxe em texto.
4. **Tradução de texto**: traduzir texto de uma língua para outra.
5. **Resolução de problemas**: resolver problemas matemáticos, lógicos ou de raciocínio.

Os agentes de IA que usam LLMs podem ser encontrados em diversas áreas, como:

1. **Chatbots**: sistemas de conver